### Question 1 :
Download the fashion-MNIST dataset and plot 1 sample image for each class as shown in the grid below. Use from keras.datasets import fashion_mnist for getting the fashion mnist dataset.

In [4]:
from keras.datasets import fashion_mnist
import numpy as np
import wandb


wandb.init(project="DL_Assignment_1", name="question_1")


(x_train, y_train), (_, _) = fashion_mnist.load_data()


class_labels = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat","Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]


images_per_class = {}

for cls in np.unique(y_train):  
    index = np.where(y_train == cls)[0][0] 
    images_per_class[class_labels[cls]] = x_train[index]  


wandb.log({"Class-wise Images": [wandb.Image(img, caption=label) for label, img in images_per_class.items()]})

wandb.finish()


### Question 2
Implement a feedforward neural network which takes images from the fashion-mnist data as input and outputs a probability distribution over the 10 classes.
Your code should be flexible such that it is easy to change the number of hidden layers and the number of neurons in each hidden layer.

In [ ]:
import numpy as np

class Optimizer:
    def __init__(self, layers, optimizer_type="sgd", learning_rate=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.optimizer_type = optimizer_type
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        
        self.momentum_w = [np.zeros((layers[i], layers[i+1])) for i in range(len(layers) - 1)]
        self.momentum_b = [np.zeros((1, layers[i+1])) for i in range(len(layers) - 1)]
        
        self.velocity_w = [np.zeros((layers[i], layers[i+1])) for i in range(len(layers) - 1)]
        self.velocity_b = [np.zeros((1, layers[i+1])) for i in range(len(layers) - 1)]
        
        self.t = 0

    def sgd(self, weights, biases, gradients_w, gradients_b):
        for i in range(len(weights)):
            weights[i] -= self.learning_rate * gradients_w[i]
            biases[i] -= self.learning_rate * gradients_b[i]

    def momentum(self, weights, biases, gradients_w, gradients_b):
        for i in range(len(weights)):
            self.momentum_w[i] = self.beta1 * self.momentum_w[i] + (1 - self.beta1) * gradients_w[i]
            self.momentum_b[i] = self.beta1 * self.momentum_b[i] + (1 - self.beta1) * gradients_b[i]
            weights[i] -= self.learning_rate * self.momentum_w[i]
            biases[i] -= self.learning_rate * self.momentum_b[i]

    def nesterov(self, weights, biases, gradients_w, gradients_b):
        for i in range(len(weights)):
            prev_momentum_w = self.momentum_w[i]
            prev_momentum_b = self.momentum_b[i]
            self.momentum_w[i] = self.beta1 * self.momentum_w[i] - self.learning_rate * gradients_w[i]
            self.momentum_b[i] = self.beta1 * self.momentum_b[i] - self.learning_rate * gradients_b[i]
            weights[i] += -self.beta1 * prev_momentum_w + (1 + self.beta1) * self.momentum_w[i]
            biases[i] += -self.beta1 * prev_momentum_b + (1 + self.beta1) * self.momentum_b[i]

    def rmsprop(self, weights, biases, gradients_w, gradients_b):
        for i in range(len(weights)):
            self.velocity_w[i] = self.beta2 * self.velocity_w[i] + (1 - self.beta2) * np.square(gradients_w[i])
            self.velocity_b[i] = self.beta2 * self.velocity_b[i] + (1 - self.beta2) * np.square(gradients_b[i])
            weights[i] -= (self.learning_rate / (np.sqrt(self.velocity_w[i]) + self.epsilon)) * gradients_w[i]
            biases[i] -= (self.learning_rate / (np.sqrt(self.velocity_b[i]) + self.epsilon)) * gradients_b[i]

    def adam(self, weights, biases, gradients_w, gradients_b):
        self.t += 1
        for i in range(len(weights)):
            self.momentum_w[i] = self.beta1 * self.momentum_w[i] + (1 - self.beta1) * gradients_w[i]
            self.momentum_b[i] = self.beta1 * self.momentum_b[i] + (1 - self.beta1) * gradients_b[i]
            
            self.velocity_w[i] = self.beta2 * self.velocity_w[i] + (1 - self.beta2) * np.square(gradients_w[i])
            self.velocity_b[i] = self.beta2 * self.velocity_b[i] + (1 - self.beta2) * np.square(gradients_b[i])
            
            m_w_hat = self.momentum_w[i] / (1 - self.beta1 ** self.t)
            m_b_hat = self.momentum_b[i] / (1 - self.beta1 ** self.t)
            v_w_hat = self.velocity_w[i] / (1 - self.beta2 ** self.t)
            v_b_hat = self.velocity_b[i] / (1 - self.beta2 ** self.t)
            
            weights[i] -= (self.learning_rate / (np.sqrt(v_w_hat) + self.epsilon)) * m_w_hat
            biases[i] -= (self.learning_rate / (np.sqrt(v_b_hat) + self.epsilon)) * m_b_hat

    def nadam(self, weights, biases, gradients_w, gradients_b):
        self.t += 1
        for i in range(len(weights)):
            self.momentum_w[i] = self.beta1 * self.momentum_w[i] + (1 - self.beta1) * gradients_w[i]
            self.momentum_b[i] = self.beta1 * self.momentum_b[i] + (1 - self.beta1) * gradients_b[i]
            
            self.velocity_w[i] = self.beta2 * self.velocity_w[i] + (1 - self.beta2) * np.square(gradients_w[i])
            self.velocity_b[i] = self.beta2 * self.velocity_b[i] + (1 - self.beta2) * np.square(gradients_b[i])
            
            m_w_hat = self.momentum_w[i] / (1 - self.beta1 ** self.t)
            m_b_hat = self.momentum_b[i] / (1 - self.beta1 ** self.t)
            v_w_hat = self.velocity_w[i] / (1 - self.beta2 ** self.t)
            v_b_hat = self.velocity_b[i] / (1 - self.beta2 ** self.t)
            
            weights[i] -= (self.learning_rate / (np.sqrt(v_w_hat) + self.epsilon)) * (self.beta1 * m_w_hat + (1 - self.beta1) * gradients_w[i] / (1 - self.beta1 ** self.t))
            biases[i] -= (self.learning_rate / (np.sqrt(v_b_hat) + self.epsilon)) * (self.beta1 * m_b_hat + (1 - self.beta1) * gradients_b[i] / (1 - self.beta1 ** self.t))

    def update_parameters(self, weights, biases, gradients_w, gradients_b):
        if self.optimizer_type == "sgd" or self.optimizer_type == "vanilla":
            self.sgd(weights, biases, gradients_w, gradients_b)
        elif self.optimizer_type == "momentum":
            self.momentum(weights, biases, gradients_w, gradients_b)
        elif self.optimizer_type == "nesterov":
            self.nesterov(weights, biases, gradients_w, gradients_b)
        else:
            raise ValueError("Invalid optimizer type")


In [19]:
import math
import random
import numpy as np
from keras.datasets import fashion_mnist
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1000007)
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

class_labels = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

def to_one_hot(labels, num_classes):
    one_hot = np.zeros((len(labels), num_classes)) 
    one_hot[np.arange(len(labels)), labels] = 1    
    return one_hot

y_train = to_one_hot(y_train, 10)
y_test = to_one_hot(y_test, 10)
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)
x_train = x_train / 255.0
x_test = x_test / 255.0



def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True)) 
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def cross_entropy_loss(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred + 1e-8)) / y_true.shape[0]


def compute_accuracy(y_true, y_pred):
    true_labels = np.argmax(y_true, axis=1)
    predicted_labels = np.argmax(y_pred, axis=1)
    return np.mean(true_labels == predicted_labels) * 100


class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers
        self.weights = []
        self.biases = []

        for i in range(len(layers) - 1):
            self.weights.append(np.random.randn(layers[i], layers[i+1]) * math.sqrt(1 / layers[i]))
            self.biases.append(np.zeros((1, layers[i+1])))

    def forward(self, X):
        activations = [X]
        for i in range(len(self.weights) - 1): 
            a = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            h = sigmoid(a)
            activations.append(h)

       
        a = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        h = softmax(a)
        activations.append(h)

        return activations

    def backward(self, X, y_true, activations):
        gradients_w = [None] * len(self.weights)
        gradients_b = [None] * len(self.biases)

       
        error = activations[-1] - y_true

        for i in reversed(range(len(self.weights))):
            gradients_w[i] = np.matmul(activations[i].T, error) / X.shape[0]
            gradients_b[i] = np.sum(error, axis=0, keepdims=True) / X.shape[0]

            if i > 0:
                error = np.dot(error, self.weights[i].T) * sigmoid_derivative(activations[i])

        return gradients_w, gradients_b

    def update_parameters(self, gradients_w, gradients_b, learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * gradients_w[i]
            self.biases[i] -= learning_rate * gradients_b[i]

    def train(self, X_train, y_train,x_test,y_test,batch=128, epochs=1000, learning_rate=0.01 , offset=10,optimizer="SGD",beta1=0.9, beta2=0.999, epsilon=1e-8):
        if optimizer == "sgd":
            batch = 1
        self.optimizer = Optimizer(self.layers, optimizer, learning_rate, beta1, beta2, epsilon)
        for epoch in range(epochs):
            for i in range(len(X_train) // batch):
                lower_bound = i * batch
                upper_bound = max((i + 1) * batch,len(X_train))
                activations = self.forward(X_train[lower_bound:upper_bound])
                loss = cross_entropy_loss(y_train[lower_bound:upper_bound], activations[-1])
                gradients_w, gradients_b = self.backward(X_train[lower_bound:upper_bound], y_train[lower_bound:upper_bound], activations)
                self.optimizer.update_parameters(self.weights, self.biases, gradients_w, gradients_b)
            if epoch % offset == 0:
                test_activations = self.forward(x_test)
                accuracy = compute_accuracy(y_test, test_activations[-1])
                print(f"Epoch {epoch}, Loss: {loss:.4f}, Accuracy: {accuracy:.2f}%")

    def predict(self, X):
        activations = self.forward(X)
        return np.argmax(activations[-1], axis=1)




layers = [784,32,32,32,10]
nn = NeuralNetwork(layers)
# activation = nn.forward(x_train[:1000])
# for i in activation[-1]:
#     print(f"Probability Distribution: {i}")

In [23]:
nn.train(x_train , y_train,x_test,y_test,batch=6000, epochs=100, learning_rate=0.5,offset=1,optimizer="vanilla")

Epoch 0, Loss: 0.9670, Accuracy: 63.07%
Epoch 1, Loss: 0.9515, Accuracy: 63.89%
Epoch 2, Loss: 0.9370, Accuracy: 64.38%
Epoch 3, Loss: 0.9484, Accuracy: 61.40%
Epoch 4, Loss: 0.9346, Accuracy: 63.60%
Epoch 5, Loss: 0.9002, Accuracy: 66.03%
Epoch 6, Loss: 0.8965, Accuracy: 65.18%
Epoch 7, Loss: 0.9127, Accuracy: 63.87%
Epoch 8, Loss: 0.8723, Accuracy: 66.59%
Epoch 9, Loss: 0.8620, Accuracy: 66.76%
Epoch 10, Loss: 0.8628, Accuracy: 66.44%
Epoch 11, Loss: 0.8453, Accuracy: 67.45%
Epoch 12, Loss: 0.8300, Accuracy: 68.28%
Epoch 13, Loss: 0.8224, Accuracy: 68.41%
Epoch 14, Loss: 0.8150, Accuracy: 68.76%
Epoch 15, Loss: 0.8040, Accuracy: 69.22%
Epoch 16, Loss: 0.7939, Accuracy: 69.72%
Epoch 17, Loss: 0.7862, Accuracy: 70.00%
Epoch 18, Loss: 0.7793, Accuracy: 70.29%
Epoch 19, Loss: 0.7720, Accuracy: 70.46%
Epoch 20, Loss: 0.7645, Accuracy: 70.57%
Epoch 21, Loss: 0.7578, Accuracy: 70.75%
Epoch 22, Loss: 0.7518, Accuracy: 70.99%
Epoch 23, Loss: 0.7460, Accuracy: 71.14%
Epoch 24, Loss: 0.7403, Ac